In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
# Explainability
import shap

print("All libraries loaded successfully!")

C:\Users\indhi\dhct_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries loaded successfully!


In [2]:
import os
print(os.getcwd())  
folder_path = './ht24wk394/' 
os.listdir(folder_path)

C:\Users\indhi


['.ipynb_checkpoints',
 'activity.csv',
 'add_clinical.csv',
 'Consent_Roadmap_9Aug2024_SS_Consent_boxes_fixed.pdf',
 'daily_activity.csv',
 'daily_hr.csv',
 'daily_steps.csv',
 'data_dictionary.txt',
 'demographic_data.csv',
 'hr.csv',
 'ht24wk394_metadata_report.txt',
 'infections.csv',
 'mood.csv',
 'outcome.csv',
 'PROMIS_tscore.csv',
 'readme.txt',
 'readmissions.csv',
 'sleep_classic.csv',
 'sleep_stages.csv',
 'steps.csv']

In [3]:
import pandas as pd

# Load the files
df_activity = pd.read_csv('./ht24wk394/daily_activity.csv')
df_hr = pd.read_csv('./ht24wk394/daily_hr.csv')
df_steps = pd.read_csv('./ht24wk394/daily_steps.csv')
df_sleep = pd.read_csv('./ht24wk394/sleep_stages.csv')
df_mood = pd.read_csv('./ht24wk394/mood.csv')
print("Activity columns:", df_activity.columns)
print("HR columns:", df_hr.columns)
print("Steps columns:", df_steps.columns)
print("sleep columns:", df_sleep.columns)
print("mood columns:", df_mood.columns)


Activity columns: Index(['STUDY_PRTCPT_ID', 'Group', 'DaysFromTransplant', 'sedentary',
       'lightly_active', 'moderately_active', 'very_active',
       'total_active_minutes', 'total_measured_minutes', 'coverage',
       'percent_sedentary', 'percent_active'],
      dtype='object')
HR columns: Index(['STUDY_PRTCPT_ID', 'Group', 'DaysFromTransplant', 'mean_hr',
       'median_hr', 'min_hr', 'max_hr', 'sd_hr', 'n_measurements', 'coverage',
       'morning_hr', 'afternoon_hr', 'evening_hr', 'night_hr'],
      dtype='object')
Steps columns: Index(['STUDY_PRTCPT_ID', 'Group', 'DaysFromTransplant', 'total_steps',
       'mean_steps_per_minute', 'max_steps_per_minute', 'active_minutes',
       'n_measurements', 'coverage', 'morning_steps', 'afternoon_steps',
       'evening_steps', 'night_steps'],
      dtype='object')
sleep columns: Index(['STUDY_PRTCPT_ID', 'Group', 'DaysFromTransplant', 'SLEEP_START_TIME',
       'SLEEP_END_TIME', 'sleep_duration', 'ASLEEP_VALUE', 'INBED_VALUE',
      

In [4]:
print(f"Total Patient-Days: {len(df_activity)}")
print(f"Number of Unique Participants (Patient Group in activity dataset): {df_activity[df_activity['Group'] == 'Patients']['STUDY_PRTCPT_ID'].nunique()}")
df_activity.head()

Total Patient-Days: 25397
Number of Unique Participants (Patient Group in activity dataset): 143


,STUDY_PRTCPT_ID,Group,DaysFromTransplant,sedentary,lightly_active,moderately_active,very_active,total_active_minutes,total_measured_minutes,coverage,percent_sedentary,percent_active
0,P001,Patients,0,134,15,2,0,17,151,0.104861,88.741722,11.258278
1,P001,Patients,1,125,13,1,0,14,139,0.096528,89.928058,10.071942
2,P001,Patients,2,138,11,7,3,21,159,0.110417,86.792453,13.207547
3,P001,Patients,3,182,3,8,12,23,205,0.142361,88.780488,11.219512
4,P001,Patients,4,150,9,7,6,22,172,0.119444,87.209302,12.790698


In [6]:

print(f"Total Rows: {len(df_hr)}")
print(df_hr[df_hr['Group'] == 'Patients']['STUDY_PRTCPT_ID'].nunique())
df_hr.head()

Total Rows: 25399
143


,STUDY_PRTCPT_ID,Group,DaysFromTransplant,mean_hr,median_hr,min_hr,max_hr,sd_hr,n_measurements,coverage,morning_hr,afternoon_hr,evening_hr,night_hr
0,P001,Patients,0,97.045005,97.0,70,131,10.572375,10110,7.020833,95.287747,101.060841,107.367457,89.549618
1,P001,Patients,1,95.528684,96.0,70,128,10.597767,10023,6.960417,92.085905,105.169133,101.183373,86.245552
2,P001,Patients,2,92.967455,91.0,65,138,11.150026,9679,6.721528,92.887725,99.195347,100.245861,84.379419
3,P001,Patients,3,93.912116,93.0,70,131,9.020465,9706,6.740278,90.774485,99.910907,102.069902,86.968193
4,P001,Patients,4,96.270381,94.0,70,129,9.219601,10034,6.968056,94.741976,99.468354,104.145934,90.451996


In [7]:
print(f"Total Rows: {len(df_steps)}")
print(df_steps[df_steps['Group'] == 'Patients']['STUDY_PRTCPT_ID'].nunique())
df_steps.head()

Total Rows: 24747
143


,STUDY_PRTCPT_ID,Group,DaysFromTransplant,total_steps,mean_steps_per_minute,max_steps_per_minute,active_minutes,n_measurements,coverage,morning_steps,afternoon_steps,evening_steps,night_steps
0,P001,Patients,0,343,20.176471,84,17,17,0.011806,57,165,28,93
1,P001,Patients,1,310,22.142857,77,14,14,0.009722,68,231,11,0
2,P001,Patients,2,1021,48.619048,100,21,21,0.014583,52,886,73,10
3,P001,Patients,3,1886,82.000000,99,23,23,0.015972,0,1701,185,0
4,P001,Patients,4,1204,57.333333,94,21,21,0.014583,18,954,228,4


In [8]:
print(f"Total Rows: {len(df_sleep)}")
print(df_sleep[df_sleep['Group'] == 'Patients']['STUDY_PRTCPT_ID'].nunique())
df_sleep.head()

Total Rows: 20010
136


,STUDY_PRTCPT_ID,Group,DaysFromTransplant,SLEEP_START_TIME,SLEEP_END_TIME,sleep_duration,ASLEEP_VALUE,INBED_VALUE,DEEP_MIN,LIGHT_MIN,REM_MIN,WAKE_MIN,DEEP_COUNT,LIGHT_COUNT,REM_COUNT,WAKE_COUNT
0,P001,Patients,0,21:47:30,08:21:30,10.566667,578,634,81,325,172,56,2,33,7,33
1,P171,Caregivers,0,21:43:00,08:23:00,10.666667,579,640,101,274,204,61,3,24,10,25
2,P001,Patients,1,22:50:30,08:41:30,9.850000,529,591,89,255,185,62,3,23,12,23
3,P001,Patients,2,22:41:30,08:38:30,9.950000,519,597,70,352,97,78,3,28,10,23
4,P171,Caregivers,2,00:49:30,09:22:30,8.550000,477,513,58,267,152,36,2,22,7,22


In [77]:
print(f"Total Rows: {len(df_mood)}")
print(df_mood[df_mood['Group'] == 'Patients']['STUDY_PRTCPT_ID'].nunique())
print(df_mood.isna().sum())
df_mood.head()

Total Rows: 20196
154
STUDY_PRTCPT_ID       0
MOOD                  0
time_stamp            0
Group                 0
DaysFromTransplant    0
dtype: int64


,STUDY_PRTCPT_ID,MOOD,time_stamp,Group,DaysFromTransplant
0,P001,7,18:00:35,Patients,5
1,P270,8,16:04:45,Caregivers,1
2,P001,7,18:59:35,Patients,2
3,P270,8,22:49:29,Caregivers,3
4,P270,7,16:40:05,Caregivers,4


In [78]:
# 1. Merge  Steps
# Define the correct keys for the dHCT dataset
merge_keys = ['STUDY_PRTCPT_ID', 'DaysFromTransplant'] # avoid memory error

# 1. Start with Activity
merged_df = df_activity.copy()

# 2. Merge Steps
# (Adding suffixes for 'Group' to avoid conflicts during the merge process)
merged_df = pd.merge(merged_df, df_steps, on=merge_keys, how='left', suffixes=('', '_steps'))

# 3. Merge Heart Rate
merged_df = pd.merge(merged_df, df_hr, on=merge_keys, how='left', suffixes=('', '_hr'))

# 4. Merge Sleep
merged_df = pd.merge(merged_df, df_sleep, on=merge_keys, how='left', suffixes=('', '_sleep'))

# 5. Clean up redundant 'Group' columns (Patient vs Caregiver info is identical in all files)
cols_to_drop = [c for c in merged_df.columns if 'Group_' in c or 'Group_hr' in c or 'Group_sleep' in c]
merged_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

merged_df = pd.merge(merged_df, df_mood, on=merge_keys, how='left', suffixes=('', '_mood'))

# 4. Clean up redundant 'Group' columns if they were duplicated
if 'Group_mood' in merged_df.columns:
    merged_df.drop(columns=['Group_mood'], inplace=True)


print("✅ Merged Successfully!")
print(f"Total Columns: {len(merged_df.columns)}")
print(f"Total Rows: {len(merged_df)}")
print(merged_df.isna().sum())
print(merged_df[merged_df['Group'] == 'Patients']['STUDY_PRTCPT_ID'].nunique())
merged_df.head()

✅ Merged Successfully!
Total Columns: 48
Total Rows: 28658
STUDY_PRTCPT_ID               0
Group                         0
DaysFromTransplant            0
sedentary                     0
lightly_active                0
moderately_active             0
very_active                   0
total_active_minutes          0
total_measured_minutes        0
coverage                      0
percent_sedentary             0
percent_active                0
total_steps                 662
mean_steps_per_minute       662
max_steps_per_minute        662
active_minutes              662
n_measurements              662
coverage_steps              662
morning_steps               662
afternoon_steps             662
evening_steps               662
night_steps                 662
mean_hr                       0
median_hr                     0
min_hr                        0
max_hr                        0
sd_hr                        43
n_measurements_hr             0
coverage_hr                   0
morning_hr   

,STUDY_PRTCPT_ID,Group,DaysFromTransplant,sedentary,lightly_active,moderately_active,very_active,total_active_minutes,total_measured_minutes,coverage,...,DEEP_MIN,LIGHT_MIN,REM_MIN,WAKE_MIN,DEEP_COUNT,LIGHT_COUNT,REM_COUNT,WAKE_COUNT,MOOD,time_stamp
0,P001,Patients,0,134,15,2,0,17,151,0.104861,...,81.0,325.0,172.0,56.0,2.0,33.0,7.0,33.0,9.0,18:24:09
1,P001,Patients,1,125,13,1,0,14,139,0.096528,...,89.0,255.0,185.0,62.0,3.0,23.0,12.0,23.0,9.0,18:04:39
2,P001,Patients,2,138,11,7,3,21,159,0.110417,...,70.0,352.0,97.0,78.0,3.0,28.0,10.0,23.0,7.0,18:59:35
3,P001,Patients,3,182,3,8,12,23,205,0.142361,...,26.0,459.0,83.0,102.0,3.0,36.0,8.0,31.0,7.0,18:09:33
4,P001,Patients,4,150,9,7,6,22,172,0.119444,...,69.0,283.0,142.0,75.0,4.0,27.0,13.0,28.0,6.0,18:15:23


In [11]:
merged_df.head()
print("merged:", merged_df.columns)

merged: Index(['STUDY_PRTCPT_ID', 'Group', 'DaysFromTransplant', 'sedentary',
       'lightly_active', 'moderately_active', 'very_active',
       'total_active_minutes', 'total_measured_minutes', 'coverage',
       'percent_sedentary', 'percent_active', 'total_steps',
       'mean_steps_per_minute', 'max_steps_per_minute', 'active_minutes',
       'n_measurements', 'coverage_steps', 'morning_steps', 'afternoon_steps',
       'evening_steps', 'night_steps', 'mean_hr', 'median_hr', 'min_hr',
       'max_hr', 'sd_hr', 'n_measurements_hr', 'coverage_hr', 'morning_hr',
       'afternoon_hr', 'evening_hr', 'night_hr', 'SLEEP_START_TIME',
       'SLEEP_END_TIME', 'sleep_duration', 'ASLEEP_VALUE', 'INBED_VALUE',
       'DEEP_MIN', 'LIGHT_MIN', 'REM_MIN', 'WAKE_MIN', 'DEEP_COUNT',
       'LIGHT_COUNT', 'REM_COUNT', 'WAKE_COUNT', 'MOOD', 'time_stamp'],
      dtype='object')


In [12]:
# Create a dedicated dataframe for patients
df_patients = merged_df[merged_df['Group'] == 'Patients'].copy()
df_caregivers = merged_df[merged_df['Group'] == 'Caregivers'].copy()
print(f"Patient-specific rows: {len(df_patients)}")
print("Unique values in Group column:", merged_df['Group'].unique())
print(len(df_patients['STUDY_PRTCPT_ID'].unique()))
print(len(df_caregivers['STUDY_PRTCPT_ID'].unique()))
#print(df_patients.isna().sum())
df_patients.head()


Patient-specific rows: 13123
Unique values in Group column: ['Patients' 'Caregivers']
143
156


,STUDY_PRTCPT_ID,Group,DaysFromTransplant,sedentary,lightly_active,moderately_active,very_active,total_active_minutes,total_measured_minutes,coverage,...,DEEP_MIN,LIGHT_MIN,REM_MIN,WAKE_MIN,DEEP_COUNT,LIGHT_COUNT,REM_COUNT,WAKE_COUNT,MOOD,time_stamp
0,P001,Patients,0,134,15,2,0,17,151,0.104861,...,81.0,325.0,172.0,56.0,2.0,33.0,7.0,33.0,9.0,18:24:09
1,P001,Patients,1,125,13,1,0,14,139,0.096528,...,89.0,255.0,185.0,62.0,3.0,23.0,12.0,23.0,9.0,18:04:39
2,P001,Patients,2,138,11,7,3,21,159,0.110417,...,70.0,352.0,97.0,78.0,3.0,28.0,10.0,23.0,7.0,18:59:35
3,P001,Patients,3,182,3,8,12,23,205,0.142361,...,26.0,459.0,83.0,102.0,3.0,36.0,8.0,31.0,7.0,18:09:33
4,P001,Patients,4,150,9,7,6,22,172,0.119444,...,69.0,283.0,142.0,75.0,4.0,27.0,13.0,28.0,6.0,18:15:23


In [13]:
df_caregivers.head()

,STUDY_PRTCPT_ID,Group,DaysFromTransplant,sedentary,lightly_active,moderately_active,very_active,total_active_minutes,total_measured_minutes,coverage,...,DEEP_MIN,LIGHT_MIN,REM_MIN,WAKE_MIN,DEEP_COUNT,LIGHT_COUNT,REM_COUNT,WAKE_COUNT,MOOD,time_stamp
13123,P171,Caregivers,0,132,26,0,0,26,158,0.109722,...,101.0,274.0,204.0,61.0,3.0,24.0,10.0,25.0,10.0,09:57:37
13124,P171,Caregivers,1,123,38,0,0,38,161,0.111806,...,88.0,317.0,130.0,57.0,2.0,27.0,8.0,28.0,10.0,13:51:24
13125,P171,Caregivers,2,105,48,0,0,48,153,0.106250,...,58.0,267.0,152.0,36.0,2.0,22.0,7.0,22.0,10.0,21:06:45
13126,P171,Caregivers,3,131,62,0,0,62,193,0.134028,...,82.0,318.0,150.0,65.0,4.0,21.0,8.0,21.0,10.0,20:00:10
13127,P171,Caregivers,4,133,36,0,0,36,169,0.117361,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,10:19:53


In [14]:
df_outcomes = pd.read_csv('./ht24wk394/add_clinical.csv')
df_readmission = pd.read_csv('./ht24wk394/readmissions.csv')
print("clinical outcome cloumns:", df_outcomes.columns)
print("readmission columns:", df_readmission.columns)
print(df_readmission.isna().sum())
#print("Unique values in Group column:", df_readmission['admission_reason'].unique())
#print("Unique values in Group column:", df_outcomes['STUDY_PRTCPT_ID'].unique())


clinical outcome cloumns: Index(['STUDY_PRTCPT_ID', 'Diagnosis', 'Conditioning Regimen', 'Graft Source'], dtype='object')
readmission columns: Index(['STUDY_PRTCPT_ID', 'number_admission', 'date_admit', 'admission_reason',
       'date_discharge', 'days_in_hospital'],
      dtype='object')
STUDY_PRTCPT_ID      0
number_admission     0
date_admit          72
admission_reason    72
date_discharge      72
days_in_hospital    72
dtype: int64


In [15]:
# 1. Get the maximum admission number for each unique patient
patient_max_admissions = df_readmission.groupby('STUDY_PRTCPT_ID')['number_admission'].max()

# 2. Count how many unique patients fall into each admission bucket
admission_counts = patient_max_admissions.value_counts().sort_index()

# 3. Display the results clearly
print("Unique Patients by Total Admissions:")
print(admission_counts)


Unique Patients by Total Admissions:
number_admission
0     72
1     28
2     24
3     12
4     14
6      4
7      1
8      1
10     1
Name: count, dtype: int64


In [19]:
# Find IDs of patients who have been readmitted at least once
readmitted_ids = df_readmission.loc[df_readmission['number_admission'] >= 1, 'STUDY_PRTCPT_ID']

# Create the label directly in df_outcomes
df_outcomes['readmission_label'] = df_outcomes['STUDY_PRTCPT_ID'].isin(readmitted_ids).astype(int)

print(df_outcomes['readmission_label'].value_counts())
print(f"Total number of unique patients:{len(df_outcomes['STUDY_PRTCPT_ID'].unique())}")
df_outcomes.head()

readmission_label
1    85
0    83
Name: count, dtype: int64
Total number of unique patients:168


,STUDY_PRTCPT_ID,Diagnosis,Conditioning Regimen,Graft Source,readmission_label
0,P001,Acute myeloid leukemia,FluBu4,Peripheral Blood Stem Cells,1
1,P002,Acute lymphoblastic leukemia,FluMelTBI + ptCy,Bone Marrow,1
2,P003,Acute myeloid leukemia,FluMel,Peripheral Blood Stem Cells,1
3,P004,Myelodysplastic syndrome,FluMel,Bone Marrow,1
4,P005,Acute myeloid leukemia,FluMel,Peripheral Blood Stem Cells,1


In [198]:

print(f"Total number of rows: {len(df_readmission)}")
print(f"Total number of unique patients:{len(df_readmission['STUDY_PRTCPT_ID'].unique())}")
df_readmission.head()

Total number of rows: 289
Total number of unique patients:157


,STUDY_PRTCPT_ID,number_admission,date_admit,admission_reason,date_discharge,days_in_hospital
0,P001,1,315.0,"new diagnosis of high-grade myeloid neoplasm, ...",337.0,22.0
1,P002,1,50.0,"Fever, CMV viremia",63.0,13.0
2,P002,2,77.0,CMV viremia,107.0,30.0
3,P002,3,137.0,"nausea, vomitting, diarrhea. C diff positive",158.0,21.0
4,P002,4,172.0,septic shock; HRE and adenovirus positive. Pas...,189.0,17.0


In [199]:
admitted_before_30 = df_readmission[df_readmission['date_admit'] < 30]
patients_before_30 = df_readmission[df_readmission['date_admit'] < 30]['STUDY_PRTCPT_ID'].unique()
print(len(patients_before_30))
admitted_before_30.head()

20


,STUDY_PRTCPT_ID,number_admission,date_admit,admission_reason,date_discharge,days_in_hospital
5,P003,1,21.0,Positive wound culture/Central line infection,29.0,8.0
11,P005,1,27.0,GVHD (graft versus host disease),53.0,26.0
20,P010,1,20.0,"nausea, vomiting, diarrhea. Rapid positive COV...",23.0,3.0
24,P011,1,21.0,ED visit; bloody stool,21.0,0.0
28,P013,1,21.0,"pain, poor p.o. intake, and low-grade fever",37.0,16.0


In [200]:
print(f"Total number of rows: {len(df_outcomes)}")
print(f"Total number of unique patients:{len(df_outcomes['STUDY_PRTCPT_ID'].unique())}")
df_outcomes.head()

Total number of rows: 168
Total number of unique patients:168


,STUDY_PRTCPT_ID,Diagnosis,Conditioning Regimen,Graft Source
0,P001,Acute myeloid leukemia,FluBu4,Peripheral Blood Stem Cells
1,P002,Acute lymphoblastic leukemia,FluMelTBI + ptCy,Bone Marrow
2,P003,Acute myeloid leukemia,FluMel,Peripheral Blood Stem Cells
3,P004,Myelodysplastic syndrome,FluMel,Bone Marrow
4,P005,Acute myeloid leukemia,FluMel,Peripheral Blood Stem Cells


In [201]:
print(f"Total number of rows: {len(df_outcomes)}")
print(f"Total number of unique patients:{len(df_outcomes['STUDY_PRTCPT_ID'].unique())}")
df_outcomes.head()

Total number of rows: 168
Total number of unique patients:168


,STUDY_PRTCPT_ID,Diagnosis,Conditioning Regimen,Graft Source
0,P001,Acute myeloid leukemia,FluBu4,Peripheral Blood Stem Cells
1,P002,Acute lymphoblastic leukemia,FluMelTBI + ptCy,Bone Marrow
2,P003,Acute myeloid leukemia,FluMel,Peripheral Blood Stem Cells
3,P004,Myelodysplastic syndrome,FluMel,Bone Marrow
4,P005,Acute myeloid leukemia,FluMel,Peripheral Blood Stem Cells


In [202]:
# Find patients admitted before 30 days and Remove those patients entirely
df_readmission_filtered = df_readmission[~df_readmission['STUDY_PRTCPT_ID'].isin(patients_before_30)]
print(f"Total number of unique patients:{len(df_readmission_filtered['STUDY_PRTCPT_ID'].unique())}")

Total number of unique patients:137


137


Total remaining unique patients in targets: 148

Readmission Distribution:
READMISSION_LABEL
1    136
0     12
Name: count, dtype: int64


In [20]:
#print(merged_df['DaysFromTransplant'].unique())
print(len(merged_df['STUDY_PRTCPT_ID'].unique()))
print(merged_df[merged_df['Group'] == 'Patients']['STUDY_PRTCPT_ID'].nunique())

299
143


In [21]:
# Keep only the first 30 days for each patient

# 1. Keep only Patients and the first 30 days
df_30d = merged_df[(merged_df['Group'] == 'Patients') & 
                   (merged_df['DaysFromTransplant'] >= 0) & 
                   (merged_df['DaysFromTransplant'] <= 30)].copy()

# 2. Convert MOOD to numeric (just in case there are strings/NaNs)
df_30d['MOOD'] = pd.to_numeric(df_30d['MOOD'], errors='coerce')

print(f"Data points for analysis: {len(df_30d)}")
df_30d.head()
print("Number of Unique values(Patients):", len(df_30d['STUDY_PRTCPT_ID'].unique()))
#print(len(df_30d['STUDY_PRTCPT_ID'].unique()))
#print(df_30d['STUDY_PRTCPT_ID'].value_counts())
#print(df_30d.isna().sum())
#print(df_30d [df_30d ['Group'] == 'Patients']['STUDY_PRTCPT_ID'].nunique())
df_30d.head()

Data points for analysis: 4056
Number of Unique values(Patients): 140


,STUDY_PRTCPT_ID,Group,DaysFromTransplant,sedentary,lightly_active,moderately_active,very_active,total_active_minutes,total_measured_minutes,coverage,...,DEEP_MIN,LIGHT_MIN,REM_MIN,WAKE_MIN,DEEP_COUNT,LIGHT_COUNT,REM_COUNT,WAKE_COUNT,MOOD,time_stamp
0,P001,Patients,0,134,15,2,0,17,151,0.104861,...,81.0,325.0,172.0,56.0,2.0,33.0,7.0,33.0,9.0,18:24:09
1,P001,Patients,1,125,13,1,0,14,139,0.096528,...,89.0,255.0,185.0,62.0,3.0,23.0,12.0,23.0,9.0,18:04:39
2,P001,Patients,2,138,11,7,3,21,159,0.110417,...,70.0,352.0,97.0,78.0,3.0,28.0,10.0,23.0,7.0,18:59:35
3,P001,Patients,3,182,3,8,12,23,205,0.142361,...,26.0,459.0,83.0,102.0,3.0,36.0,8.0,31.0,7.0,18:09:33
4,P001,Patients,4,150,9,7,6,22,172,0.119444,...,69.0,283.0,142.0,75.0,4.0,27.0,13.0,28.0,6.0,18:15:23


In [22]:

df_30d.columns


Index(['STUDY_PRTCPT_ID', 'Group', 'DaysFromTransplant', 'sedentary',
       'lightly_active', 'moderately_active', 'very_active',
       'total_active_minutes', 'total_measured_minutes', 'coverage',
       'percent_sedentary', 'percent_active', 'total_steps',
       'mean_steps_per_minute', 'max_steps_per_minute', 'active_minutes',
       'n_measurements', 'coverage_steps', 'morning_steps', 'afternoon_steps',
       'evening_steps', 'night_steps', 'mean_hr', 'median_hr', 'min_hr',
       'max_hr', 'sd_hr', 'n_measurements_hr', 'coverage_hr', 'morning_hr',
       'afternoon_hr', 'evening_hr', 'night_hr', 'SLEEP_START_TIME',
       'SLEEP_END_TIME', 'sleep_duration', 'ASLEEP_VALUE', 'INBED_VALUE',
       'DEEP_MIN', 'LIGHT_MIN', 'REM_MIN', 'WAKE_MIN', 'DEEP_COUNT',
       'LIGHT_COUNT', 'REM_COUNT', 'WAKE_COUNT', 'MOOD', 'time_stamp'],
      dtype='object')

In [25]:
import pandas as pd
import numpy as np

# ───────────────────────────────────────────────────────────
# STEP 1: Strict Temporal Grouping & Alignment
# ───────────────────────────────────────────────────────────
# Ensure chronological alignment per patient for rolling operations
df_30d = df_30d.sort_values(['STUDY_PRTCPT_ID', 'DaysFromTransplant']).reset_index(drop=True)


df_30d['step_volatility_7d'] = (
    df_30d.groupby('STUDY_PRTCPT_ID')['total_steps']
    .transform(lambda x: x.rolling(window=7, min_periods=3).std())
    / 
    (df_30d.groupby('STUDY_PRTCPT_ID')['total_steps']
    .transform(lambda x: x.rolling(window=7, min_periods=3).mean()) + 1e-5) # Add small epsilon to avoid division by zero
)


# ───────────────────────────────────────────────────────────
# STEP 4: Sleep Architecture & Consistency (Fixed Missing Features)
# ───────────────────────────────────────────────────────────
# Bounded clean efficiency ratio
df_30d['sleep_efficiency'] = (df_30d['ASLEEP_VALUE'] / (df_30d['INBED_VALUE'] + 1)).clip(0, 1) # check this 




# Formula: (Day Mean - Night Mean) / Day Mean
df_30d['circadian_blunting_index'] = (df_30d['mean_hr'] - df_30d['night_hr']) / (df_30d['mean_hr'] + 1)
# ───────────────────────────────────────────────────────────
# STEP 6: Execution Report Verification
# ───────────────────────────────────────────────────────────
print("Upgraded Features Successfully Engineered without Errors.")
print("\n--- Feature Matrix Quick-Look ---")
display_cols = [
    'sleep_efficiency',
    'step_volatility_7d',
    'circadian_blunting_index'     
]
print(df_30d[display_cols].head())

print("\nCompleted")
print(f"Total rows: {len(df_30d)} | Total Columns in df_30d: {len(df_30d.columns)}")
print("Number of Unique values(Patients):", len(df_30d['STUDY_PRTCPT_ID'].unique()))
df_30d.head()
df_30d.columns

Upgraded Features Successfully Engineered without Errors.

--- Feature Matrix Quick-Look ---
   sleep_efficiency  step_volatility_7d  circadian_blunting_index
0          0.910236                 NaN                  0.076448
1          0.893581                 NaN                  0.096170
2          0.867893            0.719192                  0.091394
3          0.846498            0.831963                  0.073162
4          0.866667            0.688961                  0.059817

Completed
Total rows: 4056 | Total Columns in df_30d: 51
Number of Unique values(Patients): 140


Index(['STUDY_PRTCPT_ID', 'Group', 'DaysFromTransplant', 'sedentary',
       'lightly_active', 'moderately_active', 'very_active',
       'total_active_minutes', 'total_measured_minutes', 'coverage',
       'percent_sedentary', 'percent_active', 'total_steps',
       'mean_steps_per_minute', 'max_steps_per_minute', 'active_minutes',
       'n_measurements', 'coverage_steps', 'morning_steps', 'afternoon_steps',
       'evening_steps', 'night_steps', 'mean_hr', 'median_hr', 'min_hr',
       'max_hr', 'sd_hr', 'n_measurements_hr', 'coverage_hr', 'morning_hr',
       'afternoon_hr', 'evening_hr', 'night_hr', 'SLEEP_START_TIME',
       'SLEEP_END_TIME', 'sleep_duration', 'ASLEEP_VALUE', 'INBED_VALUE',
       'DEEP_MIN', 'LIGHT_MIN', 'REM_MIN', 'WAKE_MIN', 'DEEP_COUNT',
       'LIGHT_COUNT', 'REM_COUNT', 'WAKE_COUNT', 'MOOD', 'time_stamp',
       'step_volatility_7d', 'sleep_efficiency', 'circadian_blunting_index'],
      dtype='object')

In [63]:
from scipy.stats import linregress

def calculate_slope(series):
    if len(series.dropna()) < 5:  # Require at least 5 days of data
        return 0
    return linregress(range(len(series)), series.fillna(method='ffill'))[0]

# Compute step trend per patient
df_trends = df_30d.groupby('STUDY_PRTCPT_ID')['total_steps'].apply(calculate_slope).reset_index(name='step_trend_slope')

# Add to your master features list: A positive slope = recovering; negative slope = deteriorating.
df_trends.head()

C:\Users\indhi\AppData\Local\Temp\ipykernel_62652\1493171859.py:6: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return linregress(range(len(series)), series.fillna(method='ffill'))[0]


,STUDY_PRTCPT_ID,step_trend_slope
0,P001,1.757148
1,P002,3.451017
2,P003,-1030.039526
3,P004,-62.580662
4,P005,83.620321


In [64]:
# ─────────────────────────────────────────────────────────────
# STEP 1: Aggregate longitudinal → one row per patient (UPDATED)
# ─────────────────────────────────────────────────────────────

agg_features = {
    # Physical recovery
    'total_steps':           ['mean', 'sum'],
    'percent_active':        ['mean'],
    'moderately_active':     ['mean'],
    'morning_steps':         ['mean'],
    'mean_steps_per_minute': ['mean'],
    
    # Cardiovascular Stress & Adaptability
    'night_hr':              ['mean', 'std'],  # std captures circadian instability
    'mean_hr':               ['mean'],
    'circadian_blunting_index':['mean', 'min'],
    'step_volatility_7d':['max'],
    # max captures the steepest continuous vital climbing
    
    # Sleep Architecture & Disruption
    'sleep_efficiency':      ['mean', 'min','std'],
    'ASLEEP_VALUE':['max'],
    
    # Psychological & Systemic Strain
    'MOOD':                  ['mean', 'std'],
    
    
    # Temporal context
    'DaysFromTransplant':    ['max'],
}

# Aggregate the longitudinal rows
df_agg = df_30d.groupby('STUDY_PRTCPT_ID').agg(agg_features)
df_agg.columns = ['_'.join(col).strip() for col in df_agg.columns]
df_agg = df_agg.reset_index()

# Merge readmission labels cleanly
df_model_ready1 = df_agg.merge(df_outcomes[['STUDY_PRTCPT_ID', 'readmission_label']],
                               on='STUDY_PRTCPT_ID', how='inner')

print(f"Patients after aggregation : {len(df_model_ready1)}")
print(f"Class distribution:\n{df_model_ready1['readmission_label'].value_counts()}")
df_model_ready1.head()
df_model_ready1.columns

Patients after aggregation : 140
Class distribution:
readmission_label
0    71
1    69
Name: count, dtype: int64


Index(['STUDY_PRTCPT_ID', 'total_steps_mean', 'total_steps_sum',
       'percent_active_mean', 'moderately_active_mean', 'morning_steps_mean',
       'mean_steps_per_minute_mean', 'night_hr_mean', 'night_hr_std',
       'mean_hr_mean', 'circadian_blunting_index_mean',
       'circadian_blunting_index_min', 'step_volatility_7d_max',
       'sleep_efficiency_mean', 'sleep_efficiency_min', 'sleep_efficiency_std',
       'ASLEEP_VALUE_max', 'MOOD_mean', 'MOOD_std', 'DaysFromTransplant_max',
       'readmission_label'],
      dtype='object')

In [32]:
print("Number of Unique values(Patients in df_agg):", len(df_agg['STUDY_PRTCPT_ID'].unique()))
print("Number of Unique values(Patients in df_model_ready1):", len(df_model_ready1['STUDY_PRTCPT_ID'].unique()))
print("Unique patients in df_agg:",
      df_agg['STUDY_PRTCPT_ID'].nunique())

print("Unique patients in df_targets:",
      df_outcomes['STUDY_PRTCPT_ID'].nunique())
agg_ids = set(df_agg['STUDY_PRTCPT_ID'])
outcomes_ids = set(df_outcomes['STUDY_PRTCPT_ID'])

missing_in_agg = outcomes_ids - agg_ids
missing_in_outcomes = agg_ids - outcomes_ids

print("Patients in outcomes but missing in agg:",
      len(missing_in_agg))

print("Patients in agg but missing in targets:",
      len(missing_in_outcomes))

Number of Unique values(Patients in df_agg): 140
Number of Unique values(Patients in df_model_ready1): 140
Unique patients in df_agg: 140
Unique patients in df_targets: 168
Patients in outcomes but missing in agg: 28
Patients in agg but missing in targets: 0


In [34]:
common_ids = set(df_outcomes['STUDY_PRTCPT_ID']).intersection(
    set(df_agg['STUDY_PRTCPT_ID'])
)

print("Number of common unique patient IDs:", len(common_ids))

Number of common unique patient IDs: 140


In [70]:
# ─────────────────────────────────────────────────────────────
# STEP 2: Define Features and Target (UPDATED)
# ─────────────────────────────────────────────────────────────

features_list = [
    'total_steps_sum',
    #'night_hr_std',
    #'mean_hr_mean',
    'sleep_efficiency_min',
    'circadian_blunting_index_min',
    'step_volatility_7d_max',
    #'MOOD_std'
]

# Safeguard check to ensure columns exist in data matrix
features_list = [f for f in features_list if f in df_model_ready1.columns]
print(f"\nFeatures going into model : {len(features_list)}") 
missing = [f for f in features_list if f not in df_model_ready1.columns]
print("Missing features:", missing)


Features going into model : 4
Missing features: []


In [71]:
print(features_list)
print(len(features_list))

['total_steps_sum', 'sleep_efficiency_min', 'circadian_blunting_index_min', 'step_volatility_7d_max']
4


sleep_efficiency_min: Captures worst recovery state: In BMT patients, clinical deterioration often first manifests as “poor nights” characterized by fragmented and low-efficiency sleep; the minimum Sleep Efficiency Index captures this most disrupted physiological state, which may serve as an early warning signal prior to readmission.

circadian_blunting_index_min: Captures worst circadian disruption state: The minimum circadian blunting index is computed from
Circadian Blunting Index=(mean_hr−night_hr)/mean_hr+1
and then taking the minimum value across the monitoring period, which identifies the day-night period with the smallest physiologic separation (i.e., greatest circadian disruption).Thismeasures how strongly heart rate drops at night compared to daytime. So it reflects circadian (day–night) autonomic separation.


step_volatility_7d_max : The highest level of day-to-day variation in a patient’s step count over any 7-day period, reflecting how inconsistent their physical activity was during recovery .
The maximum 7-day step volatility is derived from
Step Volatility7d = SD(7-day steps)/( Mean(7-day steps)+10−5)
and then taking the maximum value across the monitoring period, which identifies the period of greatest fluctuation in daily physical activity. 
This reflects the most unstable behavioral recovery phase, where patients show highly inconsistent mobility patterns that may indicate physiologic stress, fatigue cycles, or intermittent clinical decline.


In [72]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# ─────────────────────────────────────────────────────────────
# STEP 1: Define Features and Target
# ─────────────────────────────────────────────────────────────

X = df_model_ready1[features_list]
y = df_model_ready1['readmission_label']

print("Shape of X:", X.shape)
print("Class distribution:\n", y.value_counts())

# ─────────────────────────────────────────────────────────────
# STEP 2: Train-Test Split
# ─────────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTrain size : {len(X_train)}")
print(f"Test size  : {len(X_test)}")

# ─────────────────────────────────────────────────────────────
# STEP 3: Penalized Logistic Regression Pipeline
# ─────────────────────────────────────────────────────────────
# penalty='l1'  → LASSO
# penalty='l2'  → Ridge
# penalty='elasticnet' → Elastic Net

penalized_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    
    ("model", LogisticRegression(
        penalty='l1',           # L1 regularization (LASSO)
        C=0.5,                  # lower C = stronger regularization
        solver='liblinear',     # required for L1
        class_weight='balanced',
        max_iter=5000,
        random_state=42
    ))
])

# ─────────────────────────────────────────────────────────────
# STEP 4: Train Model
# ─────────────────────────────────────────────────────────────

penalized_lr.fit(X_train, y_train)

# ─────────────────────────────────────────────────────────────
# STEP 5: Predictions
# ─────────────────────────────────────────────────────────────

y_probs = penalized_lr.predict_proba(X_test)[:, 1]

# Threshold tuning for better recall
threshold = 0.30

y_preds = (y_probs >= threshold).astype(int)

# ─────────────────────────────────────────────────────────────
# STEP 6: Evaluation
# ─────────────────────────────────────────────────────────────

print("\n===== Penalized Logistic Regression =====")

print("\nROC-AUC:", roc_auc_score(y_test, y_probs))
print("PR-AUC :", average_precision_score(y_test, y_probs))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_preds))

print("\nClassification Report:")
print(classification_report(y_test, y_preds))

# ─────────────────────────────────────────────────────────────
# STEP 7: Feature Importance (Coefficients)
# ─────────────────────────────────────────────────────────────

coef_df = pd.DataFrame({
    'Feature': features_list,
    'Coefficient': penalized_lr.named_steps['model'].coef_[0]
})

coef_df['Abs_Coefficient'] = np.abs(coef_df['Coefficient'])

coef_df = coef_df.sort_values(
    by='Abs_Coefficient',
    ascending=False
)

print("\n===== Feature Coefficients =====")
print(coef_df[['Feature', 'Coefficient']])

Shape of X: (140, 4)
Class distribution:
 readmission_label
0    71
1    69
Name: count, dtype: int64

Train size : 112
Test size  : 28

===== Penalized Logistic Regression =====

ROC-AUC: 0.7091836734693878
PR-AUC : 0.6591407381257005

Confusion Matrix:
[[ 1 13]
 [ 0 14]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.07      0.13        14
           1       0.52      1.00      0.68        14

    accuracy                           0.54        28
   macro avg       0.76      0.54      0.41        28
weighted avg       0.76      0.54      0.41        28


===== Feature Coefficients =====
                        Feature  Coefficient
3        step_volatility_7d_max     0.556064
1          sleep_efficiency_min     0.152581
0               total_steps_sum     0.142165
2  circadian_blunting_index_min     0.000000


In [76]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance

# 1. Define final feature list and target variable
features_list = [
    'total_steps_sum',
    'total_steps_mean',          # Captures daily average volume
    'sleep_efficiency_min',
    'sleep_efficiency_mean', # Captures baseline sleep quality
    'circadian_blunting_index_min',
    'circadian_blunting_index_mean', # Captures persistent heart rate blunting
    'step_volatility_7d_max'
]

X = df_model_ready1[features_list]
y = df_model_ready1['readmission_label']

# 2. Perform stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Calculate class balancing factor for the training subset
# scale_pos_weight = sum(negative instances) / sum(positive instances)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos = neg_count / pos_count

# 3. Initialize XGBoost with regularized constraints for small clinical data
xgb_model = xgb.XGBClassifier(
    n_estimators=60,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.7,
    scale_pos_weight=scale_pos,
    random_state=42,
    eval_metric='logloss'
)

# 4. Perform 5-Fold Cross-Validation on the TRAIN set only
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['roc_auc', 'average_precision', 'f1']
cv_results = cross_validate(
    xgb_model, X_train, y_train, cv=cv, scoring=scoring_metrics, return_train_score=True
)

# 5. Fit the final model on the entire training set to evaluate the hold-out test set
xgb_model.fit(X_train, y_train)
y_test_pred = xgb_model.predict(X_test)
y_test_proba = xgb_model.predict_proba(X_test)[:, 1]

# ───────────────────────────────────────────────────────────
# CUSTOM THRESHOLD PLACEMENT
# ───────────────────────────────────────────────────────────
custom_threshold = 0.55  # Adjust this value to shift precision/recall balances
y_test_pred = (y_test_proba >= custom_threshold).astype(int)
# ───────────────────────────────────────────────────────────

# 6. Calculate Permutation Importance on the test set using ROC-AUC drop
perm_importance = permutation_importance(
    xgb_model, X_test, y_test, scoring='roc_auc', n_repeats=10, random_state=42
)

importance_df = pd.DataFrame({
    'feature': features_list,
    'importance': perm_importance.importances_mean,
    'std': perm_importance.importances_std
}).sort_values(by='importance', ascending=False)

# 7. Print the results in your exact target format
print(f"Train size : {len(X_train)}  |  Test size : {len(X_test)}")
print(f"scale_pos_weight : {scale_pos:.2f}\n")

print("--- 5-Fold Cross-Validation (train set only) ---")
print(f"roc_auc              | train: {cv_results['train_roc_auc'].mean():.3f} ± {cv_results['train_roc_auc'].std():.3f}")
print(f"average_precision    | train: {cv_results['train_average_precision'].mean():.3f} ± {cv_results['train_average_precision'].std():.3f}")
print(f"f1                   | train: {cv_results['train_f1'].mean():.3f} ± {cv_results['train_f1'].std():.3f}\n")

print("--- Hold-out Test Performance ---")
print(f"ROC-AUC        : {roc_auc_score(y_test, y_test_proba):.3f}")
print(f"Avg Precision  : {average_precision_score(y_test, y_test_proba):.3f}")

# Generate classification report with the custom labels
report = classification_report(
    y_test, y_test_pred, target_names=['Healthy', 'Readmitted']
)
print(report)

print("\n--- Permutation Feature Importance (ROC-AUC drop) ---")
print(importance_df.to_string(index=False, formatters={'importance': '{:,.6f}'.format, 'std': '{:,.6f}'.format}))

Train size : 112  |  Test size : 28
scale_pos_weight : 1.04

--- 5-Fold Cross-Validation (train set only) ---
roc_auc              | train: 0.973 ± 0.007
average_precision    | train: 0.975 ± 0.006
f1                   | train: 0.895 ± 0.019

--- Hold-out Test Performance ---
ROC-AUC        : 0.781
Avg Precision  : 0.789
              precision    recall  f1-score   support

     Healthy       0.69      0.64      0.67        14
  Readmitted       0.67      0.71      0.69        14

    accuracy                           0.68        28
   macro avg       0.68      0.68      0.68        28
weighted avg       0.68      0.68      0.68        28


--- Permutation Feature Importance (ROC-AUC drop) ---
                      feature importance      std
       step_volatility_7d_max   0.100000 0.051234
        sleep_efficiency_mean   0.028571 0.030883
 circadian_blunting_index_min   0.028571 0.022028
circadian_blunting_index_mean   0.028061 0.056688
         sleep_efficiency_min   0.027041 0.03

--- Threshold Optimization Report ---
Threshold  | Healthy F1 | Readmit F1 | Macro F1  
--------------------------------------------------
0.40       | 0.353      | 0.718      | 0.535     
0.45       | 0.500      | 0.722      | 0.611     
0.50       | 0.609      | 0.727      | 0.668     
0.55       | 0.667      | 0.690      | 0.678     
0.60       | 0.750      | 0.667      | 0.708     
0.65       | 0.686      | 0.476      | 0.581     
0.70       | 0.722      | 0.500      | 0.611     
